**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Triton: GPU Kernels in Python

> ⚠️ **Draft — requires an NVIDIA GPU; code not executed here.** Verify on CUDA hardware before teaching; remove this banner after.

The modern successor to [CUDA C++](./CUDA_Cpp.ipynb): write GPU kernels as Python functions over *blocks* of data, and the Triton compiler handles threads, shared memory, and vectorization. The language OpenAI wrote FlashAttention's cousins in — and the fastest path from this curriculum to writing real kernels. Install: `pip install triton` (Linux + NVIDIA GPU).

## 1. Pre-requisites

[HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb) (the concepts Triton automates), [Performance Engineering](./Performance_Engineering.ipynb).

---
### 🕐 Session 1 of 3 — *The Block Programming Model* (~40 min)
**Goal:** one program instance per TILE, not per thread; a vector-add and its masking idiom.
**Builds on:** [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb). &nbsp; **Feeds into:** Session 2 (fused softmax).

---

💡 **Intuition.** CUDA makes you choreograph individual threads; Triton raises the unit to a **block program**: 'load this tile, compute on it, store it' — and the compiler maps tiles onto warps, picks vector widths, and stages shared memory. You keep the [roofline-level](./Performance_Engineering.ipynb) decisions (tile sizes, what to fuse); it sweats the CUDA-level ones.

In [ ]:
import torch, triton
import triton.language as tl

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)      # this instance's tile of indices
    mask = offs < n                                # the masking idiom: ragged edges, no branches
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

x = torch.randn(1_000_000, device="cuda"); y = torch.randn_like(x)
out = torch.empty_like(x)
add_kernel[(triton.cdiv(x.numel(), 1024),)](x, y, out, x.numel(), BLOCK=1024)
assert torch.allclose(out, x + y)                  # ORACLE: must equal eager torch
print("triton add == torch add ✓")

---
### 🕐 Session 2 of 3 — *Fused Softmax* (~40 min)
**Goal:** one kernel instead of five: fusion as the intensity-raising move the roofline demands.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (a flash-attention sketch).

---

💡 **Intuition.** Eager softmax launches separate kernels for max, subtract, exp, sum, divide — each a full trip through memory (intensity ≈ 0.2 FLOP/byte: hopeless, per the [roofline](./Performance_Engineering.ipynb)). **Fusion** keeps the row in registers through all five steps: one read, one write. This is the single most common source of real-world GPU speedups, and Triton makes it a 15-line function.

In [ ]:
@triton.jit
def softmax_kernel(x_ptr, out_ptr, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    mask = offs < n_cols
    x = tl.load(x_ptr + row*n_cols + offs, mask=mask, other=-float("inf"))
    x = x - tl.max(x, 0)                           # numerically-stable softmax, all in registers
    num = tl.exp(x)
    out = num / tl.sum(num, 0)
    tl.store(out_ptr + row*n_cols + offs, out, mask=mask)

X = torch.randn(4096, 1024, device="cuda")
O = torch.empty_like(X)
softmax_kernel[(4096,)](X, O, 1024, BLOCK=1024)
assert torch.allclose(O, torch.softmax(X, 1), atol=1e-6)      # ORACLE
print("fused softmax == torch.softmax ✓")

# benchmark both (do_bench handles warmup & CUDA timing correctly)
t_triton = triton.testing.do_bench(lambda: softmax_kernel[(4096,)](X, O, 1024, BLOCK=1024))
t_torch  = triton.testing.do_bench(lambda: torch.softmax(X, 1))
print(f"triton {t_triton:.3f} ms   torch {t_torch:.3f} ms")

---
### 🕐 Session 3 of 3 — *Toward Flash Attention* (~30 min)
**Goal:** the online-softmax trick that fuses attention end-to-end — sketched with pointers.
**Builds on:** Session 2.

---

💡 **Intuition (the sketch).** Attention's memory hog is the $T \times T$ score matrix. **Flash attention** never materializes it: process K/V in tiles, maintaining for each query a *running* max, normalizer, and weighted sum — the online-softmax recurrence lets a softmax be computed in pieces without ever holding the whole row. It is [overlap-save](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) for attention: block-stream the computation, carry sufficient statistics ([sufficiency](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb)!), reconstruct the exact answer. Implementing it in Triton is a rite of passage — start from the official tutorial, and verify against `torch.nn.functional.scaled_dot_product_attention` the way this course has verified everything.

---
## Where next

- [CUDA C++](./CUDA_Cpp.ipynb) — the layer below, when you need it.
- [Performance Engineering](./Performance_Engineering.ipynb) — deciding WHAT to fuse.